In [12]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator 
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Dropout,GlobalAveragePooling2D
from tensorflow.keras.models import Model
import os


In [2]:

train_datagen = ImageDataGenerator(
  rescale=1./255,
  rotation_range=20,
  shear_range= 0.2,
  zoom_range=0.2,
)
training_set = train_datagen.flow_from_directory(
    'dataset/face_mask_dataset',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='training',        
    shuffle=True
)
test_datagen = ImageDataGenerator(rescale = 1./255,
                                  validation_split=0.2  # <-- 20% of the data will be used for validation
                                  )
validation_set = test_datagen.flow_from_directory(
    'dataset/face_mask_dataset',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='validation',    
    shuffle= False
)

Found 7553 images belonging to 2 classes.
Found 1510 images belonging to 2 classes.


In [3]:
print("Training samples:", training_set.samples)
print("Validation samples:", validation_set.samples)

Training samples: 7553
Validation samples: 1510


In [4]:
base_model =MobileNetV2(weights = 'imagenet', include_top=False, input_shape=(224,224,3))
base_model.trainable = False  # Freeze base layers

In [5]:

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)  
x = Dropout(0.3)(x)
predictions = Dense(1, activation='sigmoid')(x) 
model = Model(inputs=base_model.input, outputs=predictions)

In [6]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [7]:
history = model.fit(
    training_set,
    epochs=10,
    validation_data=validation_set
)

# Final evaluation
loss, acc = model.evaluate(validation_set)
print(f"Validation Accuracy: {acc * 100:.2f}%")



g:\ml project\ml_env\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
 81/237 ━━━━━━━━━━━━━━━━━━━━ 2:20 899ms/step - accuracy: 0.7716 - loss: 0.4724

g:\ml project\ml_env\lib\site-packages\PIL\Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


237/237 ━━━━━━━━━━━━━━━━━━━━ 242s 982ms/step - accuracy: 0.8667 - loss: 0.3108 - val_accuracy: 0.9781 - val_loss: 0.0710
Epoch 2/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 215s 906ms/step - accuracy: 0.9797 - loss: 0.0621 - val_accuracy: 0.9808 - val_loss: 0.0516
Epoch 3/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 212s 894ms/step - accuracy: 0.9839 - loss: 0.0477 - val_accuracy: 0.9828 - val_loss: 0.0436
Epoch 4/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 213s 897ms/step - accuracy: 0.9852 - loss: 0.0411 - val_accuracy: 0.9841 - val_loss: 0.0421
Epoch 5/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 214s 902ms/step - accuracy: 0.9872 - loss: 0.0391 - val_accuracy: 0.9828 - val_loss: 0.0430
Epoch 6/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 220s 927ms/step - accuracy: 0.9887 - loss: 0.0319 - val_accuracy: 0.9874 - val_loss: 0.0349
Epoch 7/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 221s 933ms/step - accuracy: 0.9905 - loss: 0.0294 - val_accuracy: 0.9894 - val_loss: 0.0311
Epoch 8/10
237/237 ━━━━━━━━━━━━━━━━━━━━ 225s 951ms/step - accuracy: 0.9911 - loss: 0.02

In [13]:

# Folder containing test images
folder_path = 'dataset/single_prediction'
files = os.listdir(folder_path)

IMG_SIZE = 224  # MobileNetV2 input size

# Directly use training_set.class_indices
class_indices = training_set.class_indices
# Example: {'with_mask': 0, 'without_mask': 1}
# Invert to list to access by index
class_labels = list(class_indices.keys())

for file_name in files:
    file_path = os.path.join(folder_path, file_name)

    test_image = image.load_img(file_path, target_size=(IMG_SIZE, IMG_SIZE))
    test_image = image.img_to_array(test_image) / 255.0
    test_image = np.expand_dims(test_image, axis=0)

    result = model.predict(test_image)
    predicted_class = int(result[0][0] > 0.5)  # 0 or 1

    prediction = class_labels[predicted_class]

    print(f"{file_name} → {prediction}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
with_or_without1.jpeg → without_mask
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
with_or_without2.jpeg → with_mask


In [11]:
print(training_set.class_indices)


{'with_mask': 0, 'without_mask': 1}


Camera

In [15]:
import cv2
import numpy as np
from tensorflow.keras.preprocessing import image

IMG_SIZE = 224  # MobileNetV2 input size

# Class labels
class_labels = list(training_set.class_indices.keys())

# Start webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Convert frame (BGR -> RGB), resize, normalize
    img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img.astype("float32") / 255.0
    img = np.expand_dims(img, axis=0)

    # Predict
    result = model.predict(img, verbose=0)
    predicted_class = int(result[0][0] > 0.5)
    label = class_labels[predicted_class]

    # Display prediction on frame
    color = (0, 255, 0) if label == "with_mask" else (0, 0, 255)
    cv2.putText(frame, f"{label}", (30, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)

    cv2.imshow("Mask Detection", frame)

    # Exit on 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
